In [ ]:
import genomepy

# Proveedor sugerido: UCSC (es el estándar más compatible con la nomenclatura 'chr')
provider = "UCSC" 
genome_name = "hg38"

print(f"Iniciando descarga del genoma {genome_name} desde {provider}...")# Esta función descarga, descomprime y guarda el genoma en el directorio correcto

# Esta función descarga, descomprime y guarda el genoma en el directorio correcto
genomepy.install_genome(genome_name, provider)

print("¡Descarga de genoma completada!")

SyntaxError: unterminated string literal (detected at line 7) (593326350.py, line 7)

In [ ]:
import pandas as pd
import celloracle as co
import os
from celloracle import motif_analysis as ma

co.__version__
print("CellOracle cargado correctamente.")

# 1. Cargar y Limpiar Conexiones de Cicero
cicero_path = "/home/suaria/JAGUAR/GRN/3_output/cicero_mono_m3/monocitos_coaccess_cicero.csv"
print(f"¿Existe el archivo de Cicero?: {os.path.exists(cicero_path)}")

cicero_connections = pd.read_csv(cicero_path)
print(f"Conexiones de Cicero cargadas: {cicero_connections.shape[0]} arcos.")

# --- LA CLAVE MAGISTRAL: Normalizar Cicero a guiones bajos ---
print("Normalizando formato de Cicero (reemplazando '-' por '_')...")
cicero_connections['Peak1'] = cicero_connections['Peak1'].str.replace('-', '_')
cicero_connections['Peak2'] = cicero_connections['Peak2'].str.replace('-', '_')
# -----------------------------------------------------------

# 2. Cargar y Limpiar Picos
peaks_path = "/home/suaria/JAGUAR/GRN/2_data/peaks.csv"
peaks_df = pd.read_csv(peaks_path)
print(f"Columnas en archivo de picos: {peaks_df.columns.tolist()}")

peaks_df = ma.check_peak_format(peaks_df, ref_genome="hg38")
peaks_list = peaks_df['peak_id'].tolist()
print(f"Ejemplo de pico normalizado: {peaks_list[0]}")

# 3. Anotar Picos con información de Promotores (TSS)
print("Anotando picos con TSS...")
tss_annotated = ma.get_tss_info(peak_str_list=peaks_list, ref_genome="hg38")

# 4. FUSIÓN: Integrar TSS con Cicero
# Ahora sí, ambos archivos hablan el mismo idioma (chr1_100_200)
print("Integrando Cicero con TSS...")
integrated = ma.integrate_tss_peak_with_cicero(
    tss_peak=tss_annotated, 
    cicero_connections=cicero_connections
)

print(f"Filas tras integración: {integrated.shape[0]}")
print(integrated.head())

# 5. Crear el objeto TFinfo usando los datos integrados
print("Creando objeto TFinfo con datos integrados...")
tfi = ma.TFinfo(
    peak_data_frame=integrated,
    ref_genome="hg38"
)

# 6. Cargar y Filtrar base de datos de motivos
print("Cargando base de datos de motivos...")
motifs = ma.load_motifs("CisBP_ver2_Homo_sapiens.pfm")

# Filtrado de motivos corruptos
bad_motifs = ['M01813_2.00'] 
motifs_filtrados = [m for m in motifs if m.id not in bad_motifs]
print(f"Limpieza: {len(motifs) - len(motifs_filtrados)} motivo(s) problemático(s) eliminado(s).")

# 7. Escaneo de Motivos (El paso pesado)
print("Iniciando escaneo de motivos (esto puede tomar unos minutos)...")
tfi.scan(fpr=0.02, motifs=motifs_filtrados, verbose=True)

CellOracle cargado correctamente.
¿Existe el archivo de Cicero?: True
Conexiones de Cicero cargadas: 3934716 arcos.
Normalizando formato de Cicero (reemplazando '-' por '_')...
Columnas en archivo de picos: ['peak_id']
Normalizando formato de Picos...
Peaks before filtering:  209916
Peaks with invalid chr_name:  109
Peaks with invalid length:  0
Peaks after filtering:  209807
Ejemplo de pico normalizado: chr1_10046_10341
Anotando picos con TSS...
que bed peaks: 209807
tss peaks in que: 23289
Integrando Cicero con TSS...
Filas tras integración: 582511
                     peak_id gene_short_name  coaccess
0  chr10_100006179_100006728         BLOC1S2  0.206974
1  chr10_100006179_100006728           DNMBP  0.087562
2  chr10_100006179_100006728          ERLIN1  0.026258
3  chr10_100006179_100006728        OLMALINC  0.467977
4  chr10_100006179_100006728             SCD  0.232595
Creando objeto TFinfo con datos integrados...
Cargando base de datos de motivos...
Limpieza: 1 motivo(s) problemá

2026-07-01 04:35:36,171 - DEBUG - using background: genome hg38 with size 200


Calculating FPR-based threshold. This step may take substantial time when you load a new ref-genome. It will be done quicker on the second time. 



2026-07-01 04:35:52,340 - DEBUG - determining FPR-based threshold


Motif scan started .. It may take long time.



Scanning:   0%|          | 0/109288 [00:00<?, ? sequences/s]

In [4]:
dir(tfi)
tfi.to_dictionary()

NameError: name 'tfi' is not defined

In [ ]:
# 7. Make the final base GRN for CellOracle
print("Convirtiendo resultados a DataFrame...")
base_GRN = tfi.to_dataframe()

In [ ]:

# 8. Save the base GRN to a Parquet file for future use
tfi.save_as_parquet("Base_GRN_Monocitos_JAGUAR.parquet")
print("¡Base GRN personalizada creada y guardada con éxito!")

FileNotFoundError: [Errno 2] No such file or directory: '/root/data_folder/Base_GRN_Monocitos_JAGUAR.parquet'